# 07 Q4: สถานีใหม่สร้างผู้โดยสารใหม่ หรือแค่ย้ายมาจากสถานีเดิม

Pain point ของ BART: การขยายเส้นทางใช้งบหลายพันล้าน ต้องรู้ว่าได้ผู้โดยสารใหม่จริงเท่าไร

กรณีที่ดู
1. eBART: Antioch (ANTC) และ Pittsburg Center (PCTR) เปิด 2018-05-26 ต่อจากปลายสายเดิมที่ Pittsburg/Bay Point (PITT)
2. Warm Springs (WARM) เปิด 2017-03-25 ต่อจากปลายสายเดิมที่ Fremont (FRMT)
3. Silicon Valley: Milpitas (MLPT) และ Berryessa (BERY) เปิด 2020-06-13 ต่อจาก WARM

กรณี 1 และ 2 เกิดก่อน COVID จึงเป็นหลักฐานหลัก ส่วนกรณี 3 เปิดตรงกับช่วง COVID ผลจึงปนกันมาก ใช้ได้แค่ประกอบ

วิธีคิด
- กลุ่มสถานีปลายสาย = สถานีใหม่ + สถานีเดิมใกล้ๆ ที่ผู้โดยสารน่าจะย้ายมา
- ใช้ entries (คนเข้าสถานี) เฉลี่ยต่อวันทำงาน normal ช่วง 12 เดือนก่อนเปิด เทียบกับ 12 เดือนหลังเปิด (ข้ามสัปดาห์แรกหลังเปิด)
- ปรับด้วยการเปลี่ยนแปลงของสถานีที่เหลือทั้งระบบในช่วงเดียวกัน: ถ้าไม่มีสถานีใหม่ กลุ่มนี้น่าจะเปลี่ยนไปในอัตราเดียวกับส่วนที่เหลือของระบบ
- ผู้โดยสารใหม่สุทธิ = entries ของกลุ่มหลังเปิด - entries ของกลุ่มก่อนเปิด × (ส่วนที่เหลือหลังเปิด / ส่วนที่เหลือก่อนเปิด)
- ที่เหลือของ entries ที่สถานีใหม่ ถือว่าย้ายมาจากสถานีเดิมในกลุ่ม

ข้อจำกัด
- คนที่ย้ายมาจากสถานีนอกกลุ่ม (เช่น CONC) จะถูกนับรวมในส่วนที่เหลือของระบบ ทำให้ผู้โดยสารใหม่สุทธิอาจสูงเกินจริงเล็กน้อย
- ใช้ entries อย่างเดียว ไม่นับทริปขากลับซ้ำ
- สมมติว่าส่วนที่เหลือของระบบเป็นตัวแทนของแนวโน้มที่กลุ่มนี้จะเป็นถ้าไม่มีสถานีใหม่

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import pandas as pd
import plotly.graph_objects as go

from bart import viz
from bart.config import create_views, duck, fetch_row
from bart.perf import stage

viz.notebook_mode()

con = duck()
create_views(con)

con.execute("""
    CREATE OR REPLACE TEMP TABLE workday_entries AS
    SELECT d.station::VARCHAR AS station, d.date, d.entries
    FROM station_day AS d
    JOIN calendar AS c ON c.date = d.date
    WHERE c.day_status = 'normal' AND c.is_workday
""")

ฟังก์ชันสำหรับวัดแต่ละกรณี ใช้ซ้ำทั้ง 3 กรณี

In [2]:
def sql_list(stations):
    quoted = []
    for station in stations:
        quoted.append(f"'{station}'")
    return ", ".join(quoted)


def entries_by_station(group, before, after):
    return con.sql(f"""
        WITH labelled AS (
            SELECT
                station,
                CASE
                    WHEN date BETWEEN DATE '{before[0]}' AND DATE '{before[1]}' THEN 'before'
                    WHEN date BETWEEN DATE '{after[0]}' AND DATE '{after[1]}' THEN 'after'
                END AS period,
                entries
            FROM workday_entries
            WHERE station IN ({sql_list(group)})
        )
        SELECT
            station,
            round(avg(entries) FILTER (WHERE period = 'before')) AS before,
            round(avg(entries) FILTER (WHERE period = 'after'))  AS after
        FROM labelled
        GROUP BY station
        ORDER BY station
    """).df()


def measure_opening(name, group, new_stations, before, after):
    group_sql = sql_list(group)
    new_sql = sql_list(new_stations)

    group_before, group_after, rest_before, rest_after, new_after = fetch_row(con, f"""
        WITH labelled AS (
            SELECT
                station,
                date,
                entries,
                CASE
                    WHEN date BETWEEN DATE '{before[0]}' AND DATE '{before[1]}' THEN 'before'
                    WHEN date BETWEEN DATE '{after[0]}' AND DATE '{after[1]}' THEN 'after'
                END AS period
            FROM workday_entries
        ),
        daily AS (
            SELECT
                period,
                date,
                sum(entries) FILTER (WHERE station IN ({group_sql}))     AS group_entries,
                sum(entries) FILTER (WHERE station NOT IN ({group_sql})) AS rest_entries,
                sum(entries) FILTER (WHERE station IN ({new_sql}))       AS new_entries
            FROM labelled
            WHERE period IS NOT NULL
            GROUP BY period, date
        )
        SELECT
            avg(group_entries) FILTER (WHERE period = 'before'),
            avg(group_entries) FILTER (WHERE period = 'after'),
            avg(rest_entries) FILTER (WHERE period = 'before'),
            avg(rest_entries) FILTER (WHERE period = 'after'),
            avg(new_entries) FILTER (WHERE period = 'after')
        FROM daily
    """)

    rest_change = rest_after / rest_before
    expected_after = group_before * rest_change
    net_new = group_after - expected_after
    shifted = new_after - net_new

    return {
        "case": name,
        "group_before": round(group_before),
        "group_after": round(group_after),
        "rest_of_system_change_pct": round(100 * (rest_change - 1), 1),
        "group_expected_without_opening": round(expected_after),
        "new_station_entries": round(new_after),
        "net_new_riders": round(net_new),
        "shifted_from_old_stations": round(shifted),
        "net_new_pct_of_new_station": round(100 * net_new / new_after, 1),
    }


def monthly_entries(group, start, end):
    return con.sql(f"""
        SELECT
            date_trunc('month', date)::DATE AS month,
            station,
            avg(entries) AS entries
        FROM workday_entries
        WHERE station IN ({sql_list(group)})
          AND date BETWEEN DATE '{start}' AND DATE '{end}'
        GROUP BY month, station
        ORDER BY month, station
    """).df()


def timeline_chart(monthly, group, opened, title, subtitle, name):
    fig = go.Figure()
    for position, station in enumerate(group):
        rows = monthly[monthly["station"] == station]
        fig.add_trace(go.Scatter(
            x=rows["month"],
            y=rows["entries"],
            mode="lines",
            name=station,
            line=dict(color=viz.SERIES[position], width=3),
            hovertemplate="%{x|%b %Y}: %{y:,.0f} entries per workday<extra>" + station + "</extra>",
        ))
    viz.vline(fig, opened, "opening", y=1.02)
    fig.update_yaxes(title_text="entries per workday", tickformat="~s", rangemode="tozero")
    viz.titled(fig, title, subtitle)
    viz.save(fig, name)
    return fig


results = []

## 1. eBART (ANTC, PCTR เปิด 2018-05-26)

กลุ่ม: NCON, PITT (เดิม) + PCTR, ANTC (ใหม่)
- ก่อน: 2017-05-26 ถึง 2018-05-25
- หลัง: 2018-06-01 ถึง 2019-05-31

In [3]:
ebart_group = ["NCON", "PITT", "PCTR", "ANTC"]
ebart_before = ("2017-05-26", "2018-05-25")
ebart_after = ("2018-06-01", "2019-05-31")

with stage("07.ebart"):
    ebart_stations = entries_by_station(ebart_group, ebart_before, ebart_after)
    ebart = measure_opening("eBART 2018", ebart_group, ["PCTR", "ANTC"], ebart_before, ebart_after)
    results.append(ebart)

ebart_stations

[perf] 07.ebart: 0.01s | 2.45 cores | peak RAM 0.222 GB


,station,before,after
0,ANTC,NaN,2985.0
1,NCON,2846.0,2245.0
2,PCTR,NaN,1181.0
3,PITT,6742.0,4098.0


In [4]:
pd.Series(ebart, name="value").to_frame()

,value
case,eBART 2018
group_before,9588
group_after,10510
rest_of_system_change_pct,-1.0
group_expected_without_opening,9491
new_station_entries,4166
net_new_riders,1018
shifted_from_old_stations,3148
net_new_pct_of_new_station,24.4


In [5]:
ebart_monthly = monthly_entries(ebart_group, "2016-06-01", "2019-12-31")
timeline_chart(
    ebart_monthly,
    ebart_group,
    "2018-05-26",
    f"eBART: about {ebart['net_new_pct_of_new_station']:.0f}% of Antioch and Pittsburg Center riders are new to BART",
    "Average entries per workday by month. PITT lost riders the month ANTC and PCTR opened",
    "07_ebart_timeline",
)

ลองขยายกลุ่มให้กว้างขึ้น เพิ่ม CONC และ PHIL เข้าไป (เผื่อคนย้ายมาจากสถานีที่ไกลกว่า) ถ้าตัวเลขผู้โดยสารใหม่สุทธิใกล้เดิม แปลว่าผลไม่ได้ขึ้นกับการเลือกกลุ่มมากนัก

In [6]:
ebart_wide = measure_opening(
    "eBART 2018 (wider group)",
    ["PHIL", "CONC", "NCON", "PITT", "PCTR", "ANTC"],
    ["PCTR", "ANTC"],
    ebart_before,
    ebart_after,
)

pd.DataFrame([ebart, ebart_wide]).set_index("case")[["net_new_riders", "shifted_from_old_stations", "net_new_pct_of_new_station"]]

,net_new_riders,shifted_from_old_stations,net_new_pct_of_new_station
case,,,
eBART 2018,1018,3148,24.4
eBART 2018 (wider group),828,3339,19.9


กลุ่มกว้างได้ผู้โดยสารใหม่สุทธิน้อยลง (~20%) เพราะมีคนย้ายมาจาก CONC และ PHIL ด้วย ค่าจริงของ eBART จึงน่าจะอยู่ในช่วง 20-24% ของผู้โดยสารที่สถานีใหม่

## 2. Warm Springs (WARM เปิด 2017-03-25)

กลุ่ม: SHAY, UCTY, FRMT (เดิม) + WARM (ใหม่)
- ก่อน: 2016-03-25 ถึง 2017-03-24
- หลัง: 2017-04-01 ถึง 2018-03-31

In [7]:
warm_group = ["SHAY", "UCTY", "FRMT", "WARM"]
warm_before = ("2016-03-25", "2017-03-24")
warm_after = ("2017-04-01", "2018-03-31")

with stage("07.warm_springs"):
    warm_stations = entries_by_station(warm_group, warm_before, warm_after)
    warm = measure_opening("Warm Springs 2017", warm_group, ["WARM"], warm_before, warm_after)
    results.append(warm)

warm_stations

[perf] 07.warm_springs: 0.01s | 3.49 cores | peak RAM 0.252 GB


,station,before,after
0,FRMT,9379.0,7049.0
1,SHAY,3188.0,3160.0
2,UCTY,5034.0,4827.0
3,WARM,NaN,3075.0


In [8]:
pd.Series(warm, name="value").to_frame()

,value
case,Warm Springs 2017
group_before,17601
group_after,18111
rest_of_system_change_pct,-3.2
group_expected_without_opening,17046
new_station_entries,3075
net_new_riders,1065
shifted_from_old_stations,2010
net_new_pct_of_new_station,34.6


In [9]:
warm_monthly = monthly_entries(warm_group, "2015-09-01", "2018-12-31")
timeline_chart(
    warm_monthly,
    warm_group,
    "2017-03-25",
    f"Warm Springs: about {warm['net_new_pct_of_new_station']:.0f}% of its riders are new; the rest came mostly from Fremont",
    "Average entries per workday by month",
    "07_warm_springs_timeline",
)

ขยายกลุ่มเหมือนกัน เพิ่ม HAYW และ BAYF

In [10]:
warm_wide = measure_opening(
    "Warm Springs 2017 (wider group)",
    ["BAYF", "HAYW", "SHAY", "UCTY", "FRMT", "WARM"],
    ["WARM"],
    warm_before,
    warm_after,
)

pd.DataFrame([warm, warm_wide]).set_index("case")[["net_new_riders", "shifted_from_old_stations", "net_new_pct_of_new_station"]]

,net_new_riders,shifted_from_old_stations,net_new_pct_of_new_station
case,,,
Warm Springs 2017,1065,2010,34.6
Warm Springs 2017 (wider group),864,2212,28.1


เหมือนกับ eBART กลุ่มกว้างได้ค่าต่ำลง (~28%) ค่าจริงของ Warm Springs น่าจะอยู่ในช่วง 28-35%

## 3. Silicon Valley (MLPT, BERY เปิด 2020-06-13)

กลุ่ม: SHAY, UCTY, FRMT, WARM (เดิม) + MLPT, BERY (ใหม่)
- ก่อน: ม.ค.-ก.พ. 2020 (2 เดือนสุดท้ายก่อน COVID)
- หลัง: ม.ค.-เม.ย. 2023

ข้อจำกัดสำคัญ: ส่วนที่เหลือของระบบเหลือแค่ ~38% ของก่อน COVID ตัวเลข "ผู้โดยสารใหม่สุทธิ" ในกรณีนี้จึงปนผลของ COVID ไว้มาก
ที่พอบอกได้คือ กลุ่มนี้ฟื้นดีกว่าส่วนที่เหลือของระบบแค่ไหน และ WARM เสียผู้โดยสารมากกว่าเพื่อนบ้านไหม

In [11]:
sv_group = ["SHAY", "UCTY", "FRMT", "WARM", "MLPT", "BERY"]
sv_before = ("2020-01-01", "2020-02-29")
sv_after = ("2023-01-01", "2023-04-30")

with stage("07.silicon_valley"):
    sv_stations = entries_by_station(sv_group, sv_before, sv_after)
    sv = measure_opening("Silicon Valley 2020 *", sv_group, ["MLPT", "BERY"], sv_before, sv_after)
    results.append(sv)

recovery = []
for row in sv_stations.to_dict("records"):
    if pd.isna(row["before"]):
        recovery.append(None)
    else:
        recovery.append(round(100 * row["after"] / row["before"], 1))
sv_stations["after_pct_of_before"] = recovery

sv_stations

[perf] 07.silicon_valley: 0.01s | 3.06 cores | peak RAM 0.263 GB


,station,before,after,after_pct_of_before
0,BERY,NaN,1154.0,NaN
1,FRMT,5404.0,1744.0,32.3
2,MLPT,NaN,922.0,NaN
3,SHAY,3044.0,1327.0,43.6
4,UCTY,4521.0,1626.0,36.0
5,WARM,4196.0,1173.0,28.0


In [12]:
pd.Series(sv, name="value").to_frame()

,value
case,Silicon Valley 2020 *
group_before,17165
group_after,7945
rest_of_system_change_pct,-62.3
group_expected_without_opening,6478
new_station_entries,2076
net_new_riders,1467
shifted_from_old_stations,608
net_new_pct_of_new_station,70.7


In [13]:
sv_monthly = monthly_entries(sv_group, "2019-01-01", "2023-04-30")
timeline_chart(
    sv_monthly,
    sv_group,
    "2020-06-13",
    "Silicon Valley opened in the middle of COVID; WARM fell further than its neighbours",
    "Average entries per workday by month. The opening is mixed with the COVID drop, so this case is only supporting evidence",
    "07_silicon_valley_timeline",
)

## 4. เทียบทั้ง 3 กรณี

แบ่ง entries ของสถานีใหม่เป็น 2 ส่วน: ผู้โดยสารใหม่สุทธิ กับ ส่วนที่ย้ายมาจากสถานีเดิม

In [14]:
comparison = pd.DataFrame(results).set_index("case")
comparison

,group_before,group_after,rest_of_system_change_pct,group_expected_without_opening,new_station_entries,net_new_riders,shifted_from_old_stations,net_new_pct_of_new_station
case,,,,,,,,
eBART 2018,9588,10510,-1.0,9491,4166,1018,3148,24.4
Warm Springs 2017,17601,18111,-3.2,17046,3075,1065,2010,34.6
Silicon Valley 2020 *,17165,7945,-62.3,6478,2076,1467,608,70.7


In [15]:
case_labels = []
net_new = []
shifted = []
for row in results:
    case_labels.append(row["case"])
    net_new.append(row["net_new_riders"])
    shifted.append(row["shifted_from_old_stations"])

shifted_bars = go.Bar(name="moved from nearby old stations", y=case_labels, x=shifted, orientation="h",
                      marker_color=viz.CONTEXT, hovertemplate="%{y}: %{x:,.0f} entries per workday<extra>moved</extra>")
new_bars = go.Bar(name="net new riders", y=case_labels, x=net_new, orientation="h",
                  marker_color=viz.ACCENT, hovertemplate="%{y}: %{x:,.0f} entries per workday<extra>net new</extra>")

fig = go.Figure([new_bars, shifted_bars])
fig.update_layout(barmode="stack", hovermode="closest", margin=dict(l=180))
fig.update_xaxes(title_text="entries per workday at the new stations", tickformat="~s", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False, autorange="reversed")

title = "Only a quarter to a third of new-station riders are new to BART"
subtitle = "Entries per workday at the new stations, net new vs moved from nearby stations. * opened during COVID, not comparable"
viz.titled(fig, title, subtitle)
fig.update_layout(title_xref="container", title_x=0.02)
viz.save(fig, "07_new_vs_moved", height=500)
fig

## สรุป: Insight → Pain point

pain point ของ BART ในข้อนี้คือ การขยายเส้นทางใช้งบหลายพันล้าน ต้องรู้ว่าได้ผู้โดยสารใหม่จริงเท่าไร

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ |
|---|---|---|---|---|
| 1 | eBART: ปีแรก ANTC และ PCTR มีคนเข้าสถานี 4,166 คนต่อวันทำงาน แต่ผู้โดยสารใหม่สุทธิมีแค่ ~830-1,020 คน (20-24%) ที่เหลือย้ายมาจากสถานีเดิม PITT เสียไป 2,644 คน (-39%) ในเดือนที่เปิด | ได้ผู้โดยสารใหม่จริงเท่าไร | ตรงจุด เป็นกรณีที่เกิดก่อน COVID และผลยังใกล้เดิมเมื่อเปลี่ยนกลุ่มสถานี | เวลาประเมินโครงการขยายสาย ให้ใช้ผู้โดยสารใหม่สุทธิ ไม่ใช่ยอดคนขึ้นที่สถานีใหม่ ซึ่งสูงเกินจริงราว 4 เท่า |
| 2 | Warm Springs: ปีแรกมีคนเข้าสถานี 3,075 คนต่อวันทำงาน ผู้โดยสารใหม่สุทธิ ~860-1,070 คน (28-35%) FRMT เสียไป 2,330 คน (-25%) | ได้ผู้โดยสารใหม่จริงเท่าไร | ตรงจุด | ผลสอดคล้องกับ eBART ใช้เป็นตัวเลขอ้างอิงได้ว่า สถานีปลายสายใหม่ได้ผู้โดยสารใหม่ราว 1 ใน 4 ถึง 1 ใน 3 |
| 3 | ทั้งสองกรณี ผู้โดยสารส่วนใหญ่ของสถานีใหม่คือคนเดิมที่ย้ายจุดขึ้นรถให้ใกล้บ้านขึ้น | ประโยชน์ของการขยายสาย | บางส่วน ประโยชน์อื่น เช่น เวลาเดินทางที่สั้นลงหรือที่จอดรถที่สถานีเดิมว่างขึ้น ข้อมูลชุดนี้วัดไม่ได้ | ประเมินโครงการด้วยทั้งผู้โดยสารใหม่และประโยชน์ต่อคนเดิม และวางแผนใช้พื้นที่จอดรถหรือ feeder bus ที่สถานีปลายสายเดิมใหม่ |
| 4 | Silicon Valley: กลุ่มสถานีสาย Fremont + South Bay ฟื้นได้ 46% ของต้นปี 2020 ดีกว่าส่วนที่เหลือของระบบ (38%) และ WARM ลดลงมากที่สุดในกลุ่ม (เหลือ 28% ขณะที่เพื่อนบ้านเหลือ 32-44%) | ได้ผู้โดยสารใหม่จริงเท่าไร | บางส่วน เปิดช่วง COVID ตัวเลขผู้โดยสารใหม่สุทธิ (71%) จึงใช้ไม่ได้ แต่รูปแบบสอดคล้องกับผู้โดยสาร WARM ย้ายไป MLPT/BERY | อธิบายได้ว่าทำไม WARM ฟื้นน้อยและเปลี่ยนบทบาทตอนเช้า (Q1, Q3) อย่าตัดบริการที่ WARM โดยดูแค่ตัวเลขการฟื้นตัว ต้องดูทั้งกลุ่ม |

ข้อจำกัด
- วิธีนี้ใช้แนวโน้มของส่วนที่เหลือของระบบแทนสิ่งที่จะเกิดถ้าไม่มีสถานีใหม่ ถ้ากลุ่มปลายสายโตต่างจากส่วนอื่นอยู่แล้ว ตัวเลขจะคลาดเคลื่อน
- ใช้ entries อย่างเดียว และดูแค่ปีแรกหลังเปิด ผู้โดยสารใหม่อาจเพิ่มขึ้นในปีต่อๆ ไป
- ไม่มีข้อมูลค่าก่อสร้าง จึงคำนวณต้นทุนต่อผู้โดยสารใหม่ไม่ได้

ต่อไป: 08_m1_forecast